# PostgreSQL → Pandas Pipeline
**Course project — AI_ML_DL_CORVIT**

This notebook demonstrates three ways to pull data from PostgreSQL into Pandas,
then performs realistic analysis operations.

| # | Driver | Engine | When to use |
|---|--------|--------|-------------|
| 1 | psycopg2 | SQLAlchemy 2.0 | Legacy codebases, widest compatibility |
| 2 | psycopg3 | SQLAlchemy 2.0 | New projects, async, binary protocol |
| 3 | ADBC + PyArrow | (native) | High-performance columnar analytics |

## 0. Setup & Imports

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # make config.py importable

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
print("Pandas version:", pd.__version__)

Pandas version: 2.3.3


## 1. Step 1 — Run DB Setup (once)
This creates the database, tables, and sample data.  
Skip this cell if the database already exists.

In [2]:
# Uncomment and run ONCE to initialise the database
 import subprocess
 subprocess.run([sys.executable, "../scripts/db_setup.py"], check=True)

IndentationError: unexpected indent (1368997512.py, line 2)

## 2. Approach 1 — psycopg2 + SQLAlchemy (Legacy)

In [3]:
from scripts.fetch_psycopg2 import fetch_sales as fetch_v2, fetch_joined

df_v2 = fetch_v2()
print(f"Shape: {df_v2.shape}")
df_v2

2026-03-29 23:35:00 | INFO     | [psycopg2] Fetched 20 rows, 6 columns.


Shape: (20, 6)


,sale_id,customer_name,product,quantity,price,sale_date
0,1,Ali Khan,Sugar,50,1200.0,2026-03-01
1,2,Sara Malik,Rice,30,900.0,2026-03-02
2,3,Ahmed Raza,Wheat Flour,80,2400.0,2026-03-03
3,4,Fatima Noor,Cooking Oil,20,3800.0,2026-03-04
4,5,Usman Tariq,Sugar,45,1080.0,2026-03-05
5,6,Zara Hussain,Rice,60,1800.0,2026-03-06
6,7,Bilal Sheikh,Tea,15,750.0,2026-03-07
7,8,Hina Baig,Wheat Flour,35,1050.0,2026-03-08
8,9,Ali Khan,Cooking Oil,10,1900.0,2026-03-09
9,10,Sara Malik,Tea,25,1250.0,2026-03-10


## 3. Approach 2 — psycopg3 + SQLAlchemy (Modern)

In [4]:
from scripts.fetch_psycopg3 import fetch_sales as fetch_v3, fetch_summary

df_v3 = fetch_v3()
print(f"Shape: {df_v3.shape}")
df_v3

2026-03-29 23:35:44 | INFO     | [psycopg3] Fetched 20 rows, 6 columns.


Shape: (20, 6)


,sale_id,customer_name,product,quantity,price,sale_date
0,1,Ali Khan,Sugar,50,1200.0,2026-03-01
1,2,Sara Malik,Rice,30,900.0,2026-03-02
2,3,Ahmed Raza,Wheat Flour,80,2400.0,2026-03-03
3,4,Fatima Noor,Cooking Oil,20,3800.0,2026-03-04
4,5,Usman Tariq,Sugar,45,1080.0,2026-03-05
5,6,Zara Hussain,Rice,60,1800.0,2026-03-06
6,7,Bilal Sheikh,Tea,15,750.0,2026-03-07
7,8,Hina Baig,Wheat Flour,35,1050.0,2026-03-08
8,9,Ali Khan,Cooking Oil,10,1900.0,2026-03-09
9,10,Sara Malik,Tea,25,1250.0,2026-03-10


## 4. Approach 3 — ADBC + PyArrow (Cutting-edge)

In [5]:
from scripts.fetch_adbc import fetch_sales as fetch_adbc, fetch_to_parquet

df_adbc = fetch_adbc()
print(f"Shape: {df_adbc.shape}")
df_adbc

2026-03-29 23:35:44 | INFO     | [ADBC] Fetched 20 rows, 6 columns via Arrow.


Shape: (20, 6)


,sale_id,customer_name,product,quantity,price,sale_date
0,1,Ali Khan,Sugar,50,1200.00,2026-03-01
1,2,Sara Malik,Rice,30,900.00,2026-03-02
2,3,Ahmed Raza,Wheat Flour,80,2400.00,2026-03-03
3,4,Fatima Noor,Cooking Oil,20,3800.00,2026-03-04
4,5,Usman Tariq,Sugar,45,1080.00,2026-03-05
5,6,Zara Hussain,Rice,60,1800.00,2026-03-06
6,7,Bilal Sheikh,Tea,15,750.00,2026-03-07
7,8,Hina Baig,Wheat Flour,35,1050.00,2026-03-08
8,9,Ali Khan,Cooking Oil,10,1900.00,2026-03-09
9,10,Sara Malik,Tea,25,1250.00,2026-03-10


---
## 5. Pandas Analysis
All three DataFrames are identical in content — we use `df_v2` for the rest.

In [6]:
# Use the psycopg2 result as the working DataFrame
df = df_v2.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        20 non-null     int64  
 1   customer_name  20 non-null     object 
 2   product        20 non-null     object 
 3   quantity       20 non-null     int64  
 4   price          20 non-null     float64
 5   sale_date      20 non-null     object 
dtypes: float64(1), int64(2), object(3)
memory usage: 1.1+ KB


### 5.1 Filtering

In [7]:
# Sales where quantity > 40 units
high_volume = df[df["quantity"] > 40].reset_index(drop=True)
print(f"High-volume sales (qty > 40): {len(high_volume)} rows")
high_volume

High-volume sales (qty > 40): 8 rows


,sale_id,customer_name,product,quantity,price,sale_date
0,1,Ali Khan,Sugar,50,1200.0,2026-03-01
1,3,Ahmed Raza,Wheat Flour,80,2400.0,2026-03-03
2,5,Usman Tariq,Sugar,45,1080.0,2026-03-05
3,6,Zara Hussain,Rice,60,1800.0,2026-03-06
4,11,Ali Khan,Sugar,50,1200.0,2026-03-01
5,13,Ahmed Raza,Wheat Flour,80,2400.0,2026-03-03
6,15,Usman Tariq,Sugar,45,1080.0,2026-03-05
7,16,Zara Hussain,Rice,60,1800.0,2026-03-06


In [8]:
# Sales in March 2026 after the 5th
df["sale_date"] = pd.to_datetime(df["sale_date"])
recent = df[df["sale_date"] > "2026-03-05"]
print(f"Sales after 05-Mar-2026: {len(recent)} rows")
recent

Sales after 05-Mar-2026: 10 rows


,sale_id,customer_name,product,quantity,price,sale_date
5,6,Zara Hussain,Rice,60,1800.0,2026-03-06
6,7,Bilal Sheikh,Tea,15,750.0,2026-03-07
7,8,Hina Baig,Wheat Flour,35,1050.0,2026-03-08
8,9,Ali Khan,Cooking Oil,10,1900.0,2026-03-09
9,10,Sara Malik,Tea,25,1250.0,2026-03-10
15,16,Zara Hussain,Rice,60,1800.0,2026-03-06
16,17,Bilal Sheikh,Tea,15,750.0,2026-03-07
17,18,Hina Baig,Wheat Flour,35,1050.0,2026-03-08
18,19,Ali Khan,Cooking Oil,10,1900.0,2026-03-09
19,20,Sara Malik,Tea,25,1250.0,2026-03-10


### 5.2 Aggregation / Group By

In [9]:
# Total quantity and revenue per product
product_summary = (
    df.groupby("product")
    .agg(
        total_qty=("quantity", "sum"),
        total_revenue=("price", "sum"),
        num_sales=("sale_id", "count"),
    )
    .sort_values("total_revenue", ascending=False)
    .reset_index()
)
product_summary

,product,total_qty,total_revenue,num_sales
0,Cooking Oil,60,11400.0,4
1,Wheat Flour,230,6900.0,4
2,Rice,180,5400.0,4
3,Sugar,190,4560.0,4
4,Tea,80,4000.0,4


In [10]:
# Revenue per customer
customer_revenue = (
    df.groupby("customer_name")["price"]
    .sum()
    .sort_values(ascending=False)
    .rename("total_spent")
    .reset_index()
)
customer_revenue

,customer_name,total_spent
0,Fatima Noor,7600.0
1,Ali Khan,6200.0
2,Ahmed Raza,4800.0
3,Sara Malik,4300.0
4,Zara Hussain,3600.0
5,Usman Tariq,2160.0
6,Hina Baig,2100.0
7,Bilal Sheikh,1500.0


### 5.3 Join (SQL-side) — sales enriched with customer metadata

In [11]:
# JOIN executed in PostgreSQL; result arrives as a single DataFrame
df_joined = fetch_joined()
df_joined

2026-03-29 23:35:46 | INFO     | [psycopg2] Fetched 20 rows, 8 columns.


,sale_id,customer_name,city,segment,product,quantity,price,sale_date
0,1,Ali Khan,Lahore,Retail,Sugar,50,1200.0,2026-03-01
1,2,Sara Malik,Karachi,Wholesale,Rice,30,900.0,2026-03-02
2,3,Ahmed Raza,Islamabad,Retail,Wheat Flour,80,2400.0,2026-03-03
3,4,Fatima Noor,Lahore,Corporate,Cooking Oil,20,3800.0,2026-03-04
4,5,Usman Tariq,Faisalabad,Retail,Sugar,45,1080.0,2026-03-05
5,6,Zara Hussain,Multan,Wholesale,Rice,60,1800.0,2026-03-06
6,7,Bilal Sheikh,Rawalpindi,Corporate,Tea,15,750.0,2026-03-07
7,8,Hina Baig,Sialkot,Retail,Wheat Flour,35,1050.0,2026-03-08
8,9,Ali Khan,Lahore,Retail,Cooking Oil,10,1900.0,2026-03-09
9,10,Sara Malik,Karachi,Wholesale,Tea,25,1250.0,2026-03-10


In [12]:
# Revenue breakdown by city and segment (pivot)
if not df_joined.empty:
    pivot = df_joined.pivot_table(
        values="price",
        index="city",
        columns="segment",
        aggfunc="sum",
        fill_value=0,
    ).round(2)
    pivot

### 5.4 Derived Columns & Descriptive Statistics

In [13]:
df["unit_price"]     = (df["price"] / df["quantity"]).round(2)
df["revenue_class"]  = pd.cut(
    df["price"],
    bins=[0, 1000, 2000, float("inf")],
    labels=["Low", "Medium", "High"],
)

df[["customer_name", "product", "quantity", "price", "unit_price", "revenue_class"]]

,customer_name,product,quantity,price,unit_price,revenue_class
0,Ali Khan,Sugar,50,1200.0,24.0,Medium
1,Sara Malik,Rice,30,900.0,30.0,Low
2,Ahmed Raza,Wheat Flour,80,2400.0,30.0,High
3,Fatima Noor,Cooking Oil,20,3800.0,190.0,High
4,Usman Tariq,Sugar,45,1080.0,24.0,Medium
5,Zara Hussain,Rice,60,1800.0,30.0,Medium
6,Bilal Sheikh,Tea,15,750.0,50.0,Low
7,Hina Baig,Wheat Flour,35,1050.0,30.0,Medium
8,Ali Khan,Cooking Oil,10,1900.0,190.0,Medium
9,Sara Malik,Tea,25,1250.0,50.0,Medium


In [14]:
df[["quantity", "price", "unit_price"]].describe().round(2)

,quantity,price,unit_price
count,20.0,20.0,20.00
mean,37.0,1613.0,64.80
std,21.3,898.9,64.86
min,10.0,750.0,24.00
25%,20.0,1050.0,30.00
50%,32.5,1225.0,30.00
75%,50.0,1900.0,50.00
max,80.0,3800.0,190.00


### 5.5 Save to Parquet (via ADBC/PyArrow — production pattern)

In [15]:
# Option A: from Pandas (always works)
os.makedirs("../data", exist_ok=True)
df.to_parquet("../data/sales_enriched.parquet", index=False)
print("Saved: data/sales_enriched.parquet")

# Option B: direct Arrow path (zero-copy, faster for large tables)
fetch_to_parquet(output_path="../data/sales_raw.parquet")
print("Saved: data/sales_raw.parquet")

2026-03-29 23:35:47 | INFO     | [ADBC] Saved 20 rows to '../data/sales_raw.parquet'.


Saved: data/sales_enriched.parquet
Saved: data/sales_raw.parquet


In [16]:
# Verify round-trip: read back from Parquet
df_parquet = pd.read_parquet("../data/sales_enriched.parquet")
print(f"Parquet rows: {len(df_parquet)}")
df_parquet.head()

Parquet rows: 20


,sale_id,customer_name,product,quantity,price,sale_date,unit_price,revenue_class
0,1,Ali Khan,Sugar,50,1200.0,2026-03-01,24.0,Medium
1,2,Sara Malik,Rice,30,900.0,2026-03-02,30.0,Low
2,3,Ahmed Raza,Wheat Flour,80,2400.0,2026-03-03,30.0,High
3,4,Fatima Noor,Cooking Oil,20,3800.0,2026-03-04,190.0,High
4,5,Usman Tariq,Sugar,45,1080.0,2026-03-05,24.0,Medium


---
## 6. Performance Comparison
Benchmark the three fetch approaches on the same query.

In [17]:
import time

QUERY = "SELECT * FROM sales"
results = {}

for label, fn in [("psycopg2", fetch_v2), ("psycopg3", fetch_v3), ("ADBC", fetch_adbc)]:
    t0 = time.perf_counter()
    _ = fn(QUERY)
    elapsed = (time.perf_counter() - t0) * 1000
    results[label] = round(elapsed, 2)
    print(f"{label:10s}: {elapsed:6.2f} ms")

fastest = min(results, key=results.get)
print(f"\nFastest on this dataset: {fastest} ({results[fastest]} ms)")
print("Note: ADBC advantage grows significantly with larger datasets (100k+ rows).")

2026-03-29 23:35:48 | INFO     | [psycopg2] Fetched 20 rows, 6 columns.
2026-03-29 23:35:48 | INFO     | [psycopg3] Fetched 20 rows, 6 columns.


psycopg2  :  47.12 ms
psycopg3  :  98.66 ms


2026-03-29 23:35:48 | INFO     | [ADBC] Fetched 20 rows, 6 columns via Arrow.


ADBC      : 163.01 ms

Fastest on this dataset: psycopg2 (47.12 ms)
Note: ADBC advantage grows significantly with larger datasets (100k+ rows).


---
## Summary

| Operation | Code pattern |
|-----------|-------------|
| Filter rows | `df[df['col'] > value]` |
| Group + aggregate | `df.groupby('col').agg(...)` |
| SQL JOIN | Delegated to PostgreSQL via `fetch_joined()` |
| Save Parquet | `df.to_parquet(path)` or `fetch_to_parquet()` |
| Derived columns | `df['new'] = df['a'] / df['b']` |
| Binning | `pd.cut(df['col'], bins=[...], labels=[...])` |